In [ ]:
import os
import sys
from datetime import datetime
today = datetime.today().strftime('%Y-%m-%d')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import scanpy as sc
import anndata as ad
import hdf5plugin
from dandelion import read_h5ddl
from Bio import SeqIO

import warnings
warnings.filterwarnings("ignore", category=ad.ImplicitModificationWarning)

# Add repo path to sys path (allows to access scripts and metadata from repo)
#repo_path,_ = os.path.split(os.path.split(os.getcwd())[0])
repo_path = '/nfs/team205/lm25/thymus_projects/thymus_ageing_atlas/T_NK_compartment'
sys.path.insert(1, repo_path) 
sys.path.insert(2, '/nfs/team205/lm25/thymus_projects/thymus_ageing_atlas/General_analysis/scripts')

# Autoreload custom scripts
%load_ext autoreload
%autoreload 2

# Define paths
plots_path = f'{repo_path}/plots/'
data_path = f'{repo_path}/data/'
model_path = os.path.join(repo_path, 'models')
general_data_path = '/nfs/team205/lm25/thymus_projects/thymus_ageing_atlas/General_analysis/data'

print('Dir for plots: {}'.format(plots_path))
print('Dir for data: {}'.format(data_path))

# Formatting
from matplotlib import font_manager
font_manager.fontManager.addfont("/nfs/team205/ny1/ThymusSpatialAtlas/software/Arial.ttf")
plt.style.use('/nfs/team205/lm25/thymus_projects/thymus_ageing_atlas/General_analysis/scripts/plotting/thyAgeing.mplstyle')

# Import custom scripts
from utils import get_latest_version,update_obs,freq_by_donor
from anno_levels import get_ct_levels, get_ct_palette, age_group_levels, age_group_palette, t_nk_groupings
from plotting.utils import plot_grouped_boxplot, calc_figsize, thyAgeing_colors, get_tint_palette, get_chroma_palette, create_blend_palette

In [ ]:
# Define paths
plots_path = f'{repo_path}/plots'
data_path = f'{repo_path}/data'
model_path = os.path.join(repo_path, 'models')
general_data_path = '/nfs/team205/lm25/thymus_projects/thymus_ageing_atlas/General_analysis/data'

print('Dir for plots: {}'.format(plots_path))
print('Dir for data: {}'.format(data_path))

## Reformat 10x reference for use with Igor

In [ ]:
# Load 10x vdj reference fasta file
vdj_10x_ref_path = f"{data_path}/references/refdata-cellranger-vdj-GRCh38-alts-ensembl-7.1.0/fasta/regions.fa"

Reformat fasta to resemble Igor input files for alignment. Example entry:

In [ ]:
# Write fasta files in format required by Igor
from Bio.Seq import Seq
from Bio.SeqRecord import SeqRecord
from Bio import SeqIO
    
trbv_records = {}
trbj_records = {}
trbd_records = {}
trav_records = {}
traj_records = {}

for record in SeqIO.parse(vdj_10x_ref_path, "fasta"):
    
    if 'REGION' in record.description: # Exlcude UTRs and other non-gene regions
    
        gene = record.id.split('|')[1]
        seq = str(record.seq)
        
        if 'TRAV' in gene:
            trav_records[gene] = seq
        elif 'TRBV' in gene:
            trbv_records[gene] = seq
        elif 'TRBJ' in gene:
            trbj_records[gene] = seq
        elif 'TRAJ' in gene:
            traj_records[gene] = seq
        elif 'TRBD' in gene:
            trbd_records[gene] = seq
trbv_records = [
    SeqRecord(Seq(seq), id=seq_id, description="")
    for seq_id, seq in trbv_records.items()
]
trbj_records = [
    SeqRecord(Seq(seq), id=seq_id, description="")
    for seq_id, seq in trbj_records.items()
]
trav_records = [
    SeqRecord(Seq(seq), id=seq_id, description  ="")
    for seq_id, seq in trav_records.items()
]
traj_records = [
    SeqRecord(Seq(seq), id=seq_id, description="")
    for seq_id, seq in traj_records.items()
]
trbd_records = [
    SeqRecord(Seq(seq), id=seq_id, description="")
    for seq_id, seq in trbd_records.items()
]

SeqIO.write(trbv_records, f"{data_path}/references/refdata-cellranger-vdj-GRCh38-alts-ensembl-7.1.0/tcr_beta/genomicVs.fasta", "fasta")
SeqIO.write(trbj_records, f"{data_path}/references/refdata-cellranger-vdj-GRCh38-alts-ensembl-7.1.0/tcr_beta/genomicJs.fasta", "fasta")
SeqIO.write(trav_records, f"{data_path}/references/refdata-cellranger-vdj-GRCh38-alts-ensembl-7.1.0/tcr_alpha/genomicVs.fasta", "fasta")
SeqIO.write(traj_records, f"{data_path}/references/refdata-cellranger-vdj-GRCh38-alts-ensembl-7.1.0/tcr_alpha/genomicJs.fasta", "fasta")
SeqIO.write(trbd_records, f"{data_path}/references/refdata-cellranger-vdj-GRCh38-alts-ensembl-7.1.0/tcr_beta/genomicDs.fasta", "fasta")

Determine anchors for V/J genes: 

> Load a semicolon separated file containing the indices/offset of the CDR3 anchors for the gene(--V or --J). The index should correspond to the first letter of the cysteine (for V) or tryptophan/phenylalanin (for J) for the nucleotide sequence of the gene. Indices are 0 based.

In [ ]:
from Bio import SeqIO

def find_anchor(seq, gene_type):
    seq = str(seq).upper()
    protein = ""

    # translate
    for i in range(0, len(seq) - 2, 3):
        codon = seq[i:i+3]
        aa = {
            "TTT":"F","TTC":"F","TTA":"L","TTG":"L",
            "TCT":"S","TCC":"S","TCA":"S","TCG":"S",
            "TAT":"Y","TAC":"Y","TAA":"*","TAG":"*",
            "TGT":"C","TGC":"C","TGA":"*","TGG":"W",
            "CTT":"L","CTC":"L","CTA":"L","CTG":"L",
            "CCT":"P","CCC":"P","CCA":"P","CCG":"P",
            "CAT":"H","CAC":"H","CAA":"Q","CAG":"Q",
            "CGT":"R","CGC":"R","CGA":"R","CGG":"R",
            "ATT":"I","ATC":"I","ATA":"I","ATG":"M",
            "ACT":"T","ACC":"T","ACA":"T","ACG":"T",
            "AAT":"N","AAC":"N","AAA":"K","AAG":"K",
            "AGT":"S","AGC":"S","AGA":"R","AGG":"R",
            "GTT":"V","GTC":"V","GTA":"V","GTG":"V",
            "GCT":"A","GCC":"A","GCA":"A","GCG":"A",
            "GAT":"D","GAC":"D","GAA":"E","GAG":"E",
            "GGT":"G","GGC":"G","GGA":"G","GGG":"G"
        }.get(codon, "X")

        protein += aa

    if gene_type == "V":
        # last cysteine in protein
        pos = protein.rfind("C")
        if pos == -1:
            return None
        return pos * 3  # convert AA index to nucleotide index

    elif gene_type == "J":
        # Check all three reading frames for first W or F
        min_pos = None
        for frame in range(3):
            prot = ""
            for i in range(frame, len(seq) - 2, 3):
                codon = seq[i:i+3]
                aa = {
                    "TTT":"F","TTC":"F","TTA":"L","TTG":"L",
                    "TCT":"S","TCC":"S","TCA":"S","TCG":"S",
                    "TAT":"Y","TAC":"Y","TAA":"*","TAG":"*",
                    "TGT":"C","TGC":"C","TGA":"*","TGG":"W",
                    "CTT":"L","CTC":"L","CTA":"L","CTG":"L",
                    "CCT":"P","CCC":"P","CCA":"P","CCG":"P",
                    "CAT":"H","CAC":"H","CAA":"Q","CAG":"Q",
                    "CGT":"R","CGC":"R","CGA":"R","CGG":"R",
                    "ATT":"I","ATC":"I","ATA":"I","ATG":"M",
                    "ACT":"T","ACC":"T","ACA":"T","ACG":"T",
                    "AAT":"N","AAC":"N","AAA":"K","AAG":"K",
                    "AGT":"S","AGC":"S","AGA":"R","AGG":"R",
                    "GTT":"V","GTC":"V","GTA":"V","GTG":"V",
                    "GCT":"A","GCC":"A","GCA":"A","GCG":"A",
                    "GAT":"D","GAC":"D","GAA":"E","GAG":"E",
                    "GGT":"G","GGC":"G","GGA":"G","GGG":"G"
                }.get(codon, "X")
                prot += aa
            w = -1 # Only look for FG
            f = prot.find("FG")
            #print(f"Translated protein sequence for frame {frame} of {gene_type} gene: {prot}") # Debug print
            #print(f"Frame {frame}: W={w}, F={f}")
            # Find the first W or F in this frame
            candidates = [x for x in [w, f] if x != -1]
            if candidates:
                aa_pos = min(candidates)
                nt_pos = frame + aa_pos * 3
                if (min_pos is None) or (nt_pos < min_pos):
                    min_pos = nt_pos
        if min_pos is None:
            return None
        return min_pos
    else:
        raise ValueError("gene_type must be V or J")


def extract_anchors(fasta_file, gene_type):
    anchors = {}
    for record in SeqIO.parse(fasta_file, "fasta"):
        print(f"Processing {record.id}...")
        anchor = find_anchor(record.seq, gene_type)
        if anchor is not None:
            anchors[record.id] = anchor
    anchors = pd.DataFrame.from_dict(anchors, orient='index', columns=['anchor']).reset_index(names='gene')
    return anchors

# Save to ;-delimited csv files
extract_anchors(f"{data_path}/references/refdata-cellranger-vdj-GRCh38-alts-ensembl-7.1.0/tcr_alpha/genomicVs.fasta", "V") \
    .to_csv(f"{data_path}/references/refdata-cellranger-vdj-GRCh38-alts-ensembl-7.1.0/tcr_alpha/genomicVs_anchors.csv", index=False, sep =';')
extract_anchors(f"{data_path}/references/refdata-cellranger-vdj-GRCh38-alts-ensembl-7.1.0/tcr_alpha/genomicJs.fasta", "J") \
    .to_csv(f"{data_path}/references/refdata-cellranger-vdj-GRCh38-alts-ensembl-7.1.0/tcr_alpha/genomicJs_anchors.csv", index=False, sep =';')
extract_anchors(f"{data_path}/references/refdata-cellranger-vdj-GRCh38-alts-ensembl-7.1.0/tcr_beta/genomicVs.fasta", "V") \
    .to_csv(f"{data_path}/references/refdata-cellranger-vdj-GRCh38-alts-ensembl-7.1.0/tcr_beta/genomicVs_anchors.csv", index=False, sep =';')
extract_anchors(f"{data_path}/references/refdata-cellranger-vdj-GRCh38-alts-ensembl-7.1.0/tcr_beta/genomicJs.fasta", "J") \
    .to_csv(f"{data_path}/references/refdata-cellranger-vdj-GRCh38-alts-ensembl-7.1.0/tcr_beta/genomicJs_anchors.csv", index=False, sep =';')


## Fit model

### Load and prep data

In [ ]:
# Load data
scvi_embed_version = 'v9_2025-03-28'
adata = ad.read_h5ad(f'{data_path}/objects/rna/thyAgeing_tSplit_scvi_{scvi_embed_version}.zarr', backed = 'r')

# For multiple text columns, you can use a loop
for column in adata.obs.columns:
    if pd.api.types.is_object_dtype(adata.obs[column]):
        try:
            adata.obs[column] = adata.obs[column].str.decode('utf-8')
        except AttributeError:  # This catches columns that are not bytes type
            pass

# Add knn predictions to adata (original HTSA reference does not have uncertainties)
ct_labels = pd.read_csv(f'{general_data_path}/objects/rna/thyAgeing_all_scvi_v5_2025-10-15_curatedAnno_v11.csv', index_col = 0)
adata.obs = adata.obs.join(ct_labels)

# Add vdj data
meta_tcr = pd.read_csv(f'{data_path}/objects/rna/thyAgeing_tSplit_scvi_v9_2025-03-28_tcrab_v6.csv', index_col=0)
adata.obs = adata.obs.join(meta_tcr)

# Update metadata
from utils import get_latest_version,update_obs

latest_meta_path = get_latest_version(dir = f'{general_data_path}/metadata', file_prefix='Thymus_ageing_metadata')
latest_meta = pd.read_excel(latest_meta_path)
update_obs(adata, latest_meta, on = 'index', ignore_warning = True)

In [ ]:
# Specify cell type columns
col_cell_type_broad = 'taa_l3'
col_cell_type_fine = 'taa_l4'
col_age_group = 'age_group'
col_age_group_levels = ['infant', 'paed', 'adult', 'aged']

In [ ]:
# Load TCRab data
tcrab_info = read_h5ddl(f'{data_path}/objects/vdj/thymusAgeing_tcrabFiltered_v6_2025-04-03.h5ddl')

tcrab_info

In [ ]:
cdr3_df = tcrab_info.data[['sequence','cdr3_aa', 'v_call', 'j_call','productive', 'locus', 'umi_count']].copy().rename(columns = {'sequence':'cdr3_nt'})
cdr3_df['names'] = [n[:-9] for n in cdr3_df.index]
cdr3_df = cdr3_df.merge(adata.obs[[col_age_group, col_cell_type_broad, 'donor']].reset_index(names='names'), on = 'names', how = 'left')

cdr3_df.head()

In [ ]:
cdr3_df.to_csv(f'{data_path}/objects/vdj/thymusAgeing_tcrabFiltered_v6_2025-04-03_cdr3_df.csv.gz', index = False, compression = 'gzip')

In [ ]:
cdr3_df = pd.read_csv(f'{data_path}/objects/vdj/thymusAgeing_tcrabFiltered_v6_2025-04-03_cdr3_df.csv.gz', compression = 'gzip')

In [ ]:
cdr3_df.groupby(['donor', 'locus','productive']).size().reset_index().to_csv(f'{data_path}/analyses/vdj/igor/thymusAgeing_tcrabFiltered_v6_2025-04-03_cdr3_df_counts.csv', index = False)

In [ ]:
donor = 'TSC-CT10'
col_cell_type_broad = 'taa_l3'

cdr3_df_sub = cdr3_df[(cdr3_df['productive'] == 'F') & (cdr3_df['donor'] == donor)].drop(columns = [col_cell_type_broad, 'donor', 'age_group'])
# If multiple chains/rows per locus, keep the one with max umi_count
cdr3_df_sub = cdr3_df_sub.sort_values('umi_count', ascending=False).drop_duplicates(subset=['names', 'locus'])
cdr3_df_sub['v_call'] = cdr3_df_sub['v_call'].str.replace(r'\*0\d+', '', regex=True)
cdr3_df_sub['j_call'] = cdr3_df_sub['j_call'].str.replace(r'\*0\d+', '', regex=True)
cdr3_df_sub['cdr3_aa'] = 'C' + cdr3_df_sub['cdr3_aa'] + 'F'
cdr3_df_sub = cdr3_df_sub.pivot(index = ['names'], columns = ['locus'])
cdr3_df_sub.columns = ['_'.join(col).strip() for col in cdr3_df_sub.columns.values]
cdr3_df_sub.head()

In [ ]:
cdr3_df.loc[(cdr3_df['productive'] == 'F') & (cdr3_df['donor'] == donor) & (cdr3_df['locus'] == 'TRB')]

In [ ]:
from Bio.Seq import Seq
from Bio.SeqRecord import SeqRecord
from Bio import SeqIO

nt_seq = cdr3_df_sub['cdr3_nt_TRB'].dropna()
nt_seq_records = [SeqRecord(Seq(seq), id=idx, description="") for idx, seq in nt_seq.items()]
SeqIO.write(nt_seq_records, f"{data_path}/analyses/vdj/igor/{donor}_TRB_nonProd_cdr3_nt.fasta", "fasta")

In [ ]:
!/nfs/team205/lm25/software/igor_1-4-0/build/bin/igor -set_wd /nfs/team205/lm25/thymus_projects/thymus_ageing_atlas/T_NK_compartment/data/analyses/vdj/igor/tmp \
    -threads 4 -stdout_f /nfs/team205/lm25/thymus_projects/thymus_ageing_atlas/T_NK_compartment/data/analyses/vdj/igor/test_trb \
    -chain TRB -species human -read_seqs /nfs/team205/lm25/thymus_projects/thymus_ageing_atlas/T_NK_compartment/data/analyses/vdj/igor/T_DP_A43_TRB_cdr3_nt.fasta -align --all

In [ ]:
!/nfs/team205/lm25/software/igor_1-4-0/build/bin/igor -set_wd /nfs/team205/lm25/thymus_projects/thymus_ageing_atlas/T_NK_compartment/data/analyses/vdj/igor/tmp \
    -threads 4 -chain TRB -batch 'TSC-CT10' -species human \
    -read_seqs /nfs/team205/lm25/thymus_projects/thymus_ageing_atlas/T_NK_compartment/data/analyses/vdj/igor/TSC-CT10_TRB_nonProd_cdr3_nt.fasta -align --all -infer -output

In [ ]:
!/nfs/team205/lm25/software/igor_1-4-0/build/bin/igor -set_wd /nfs/team205/lm25/thymus_projects/thymus_ageing_atlas/T_NK_compartment/data/analyses/vdj/igor/tmp \
    -threads 4 -batch test \
    -chain TRB -species human -read_seqs /nfs/team205/lm25/thymus_projects/thymus_ageing_atlas/T_NK_compartment/data/analyses/vdj/igor/T_DP_A43_TRB_cdr3_nt.fasta -generate 1000

In [ ]:
!/nfs/team205/lm25/software/igor_1-4-0/build/bin/igor -set_wd /nfs/team205/lm25/thymus_projects/thymus_ageing_atlas/T_NK_compartment/data/analyses/vdj/igor/tmp \
    -threads 4 -stdout_f /nfs/team205/lm25/thymus_projects/thymus_ageing_atlas/T_NK_compartment/data/analyses/vdj/igor/test_trb \
    -chain TRB -batch test -species human -read_seqs /nfs/team205/lm25/thymus_projects/thymus_ageing_atlas/T_NK_compartment/data/analyses/vdj/igor/T_DP_A43_TRB_cdr3_nt.fasta -align --all -evaluate -output -scenarios 10

In [ ]:
ctoi = 'T_CD8'
donor = 'A43'
col_cell_type_broad = 'taa_l3'

cdr3_df_sub = cdr3_df[(cdr3_df[col_cell_type_broad] == ctoi) & (cdr3_df['donor'] == donor)].drop(columns = [col_cell_type_broad, 'donor', 'age_group'])
# If multiple chains/rows per locus, keep the one with max umi_count
cdr3_df_sub = cdr3_df_sub.sort_values('umi_count', ascending=False).drop_duplicates(subset=['names', 'locus'])
cdr3_df_sub['v_call'] = cdr3_df_sub['v_call'].str.replace(r'\*0\d+', '', regex=True)
cdr3_df_sub['j_call'] = cdr3_df_sub['j_call'].str.replace(r'\*0\d+', '', regex=True)
cdr3_df_sub['cdr3_aa'] = 'C' + cdr3_df_sub['cdr3_aa'] + 'F'
cdr3_df_sub = cdr3_df_sub.pivot(index = ['names'], columns = ['locus'])
cdr3_df_sub.columns = ['_'.join(col).strip() for col in cdr3_df_sub.columns.values]
cdr3_df_sub.head()

In [ ]:
from Bio.Seq import Seq
from Bio.SeqRecord import SeqRecord
from Bio import SeqIO

nt_seq = cdr3_df_sub['cdr3_nt_TRB'].dropna()
nt_seq_records = [SeqRecord(Seq(seq), id=idx, description="") for idx, seq in nt_seq.items()]
SeqIO.write(nt_seq_records, f"{data_path}/analyses/vdj/igor/{ctoi}_{donor}_TRB_cdr3_nt.fasta", "fasta")

In [ ]:
!/nfs/team205/lm25/software/igor_1-4-0/build/bin/igor -set_wd /nfs/team205/lm25/thymus_projects/thymus_ageing_atlas/T_NK_compartment/data/analyses/vdj/igor/tmp \
    -threads 4 -chain TRB -species human -batch T_CD8_A43 \
    -read_seqs /nfs/team205/lm25/thymus_projects/thymus_ageing_atlas/T_NK_compartment/data/analyses/vdj/igor/T_CD8_A43_TRB_cdr3_nt.fasta \
    -align --all

In [ ]:
data_seqs = pd.read_csv('/nfs/team205/lm25/thymus_projects/thymus_ageing_atlas/T_NK_compartment/data/analyses/vdj/igor/tmp/aligns/T_CD8_A43_indexed_CDR3s.csv', sep = ';')
data_seqs

In [ ]:
data_seqs.iloc[:,0:3]

In [ ]:
data_seqs.iloc[:,0:3].dropna().values

In [ ]:
pgen_seqs = pd.read_csv('/nfs/team205/lm25/thymus_projects/thymus_ageing_atlas/T_NK_compartment/data/analyses/vdj/igor/tmp/test_generated/generated_seqs_werr.csv', sep = ';')
pgen_seqs

In [ ]:
from sonnia.sonia import Sonia

data_seqs=cdr3_df_sub[['cdr3_aa_TRB', 'v_call_TRB', 'j_call_TRB', 'cdr3_aa_TRA', 'v_call_TRA', 'j_call_TRA']].copy()
data_seqs = data_seqs.iloc[:,0:3].dropna().values
qm = Sonia(data_seqs = data_seqs, pgen_model = '/nfs/team205/lm25/software/SONIA/sonia/default_models/human_T_beta')

In [ ]:
from sonnia.sonia import Sonia

data_seqs=cdr3_df_sub[['cdr3_aa_TRB', 'v_call_TRB', 'j_call_TRB', 'cdr3_aa_TRA', 'v_call_TRA', 'j_call_TRA']].copy()
data_seqs = data_seqs.iloc[:,0:3].dropna().values
nm = Sonia(data_seqs = data_seqs, pgen_model = '/nfs/team205/lm25/thymus_projects/thymus_ageing_atlas/T_NK_compartment/data/analyses/vdj/igor/test_trb')

In [ ]:
!awk -F'[|;]' '{print $2","$NF","$4}' /nfs/team205/lm25/thymus_projects/thymus_ageing_atlas/T_NK_compartment/data/analyses/vdj/igor/test_trb/J_gene_CDR3_anchors.csv | sed '1i gene,anchor_index,function' > /nfs/team205/lm25/thymus_projects/thymus_ageing_atlas/T_NK_compartment/data/analyses/vdj/igor/test_trb/J_gene_CDR3_anchors.csv
!awk -F'[|;]' '{print $2","$NF","$4}' /nfs/team205/lm25/thymus_projects/thymus_ageing_atlas/T_NK_compartment/data/analyses/vdj/igor/test_trb/V_gene_CDR3_anchors.csv | sed '1i gene,anchor_index,function' > /nfs/team205/lm25/thymus_projects/thymus_ageing_atlas/T_NK_compartment/data/analyses/vdj/igor/test_trb/V_gene_CDR3_anchors.csv

In [ ]:
from sonnia.sonia_paired import SoniaPaired

data_seqs=cdr3_df_sub[['cdr3_aa_TRB', 'v_call_TRB', 'j_call_TRB', 'cdr3_aa_TRA', 'v_call_TRA', 'j_call_TRA']].copy()
data_seqs = data_seqs.dropna().values
qnm = SoniaPaired(data_seqs=data_seqs, ppost_model='humanTCR')

In [ ]:
from sonnia.sonia_paired import SoniaPaired

data_seqs=cdr3_df_sub[['cdr3_aa_TRB', 'v_call_TRB', 'j_call_TRB', 'cdr3_aa_TRA', 'v_call_TRA', 'j_call_TRA']].copy()
data_seqs = data_seqs.dropna().values
qm = SoniaPaired(data_seqs=data_seqs,pgen_model='humanTCR')